# Interactive Retirement Monte Carlo Simulator
Welcome! This notebook guides you through building a probabilistic retirement projection model. Instead of assuming a static, fixed annual return, we will model market volatility using **Monte Carlo simulations** to visualize a wide spectrum of potential financial futures.

In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Set random seed for reproducibility (optional, comment out for true randomness)
#np.random.seed(42)

## 1. The Mathematics Behind the Simulation

To model portfolio growth over time, we break the timeline into discrete annual steps. Two core mathematical concepts drive our model:

### A. The Annual Growth Equation
For any given year $t$, the portfolio value $V_{t+1}$ is determined by your starting balance, your annual contribution $C$, and a random market return $R_t$:

$$V_{t+1} = (V_t + C) \times (1 + R_t)$$

### B. Modeling Market Returns with a Normal Distribution
Rather than assuming a constant return, we treat annual market returns $R_t$ as random variables drawn from a normal distribution defined by an expected mean return ($\mu$) and volatility ($\sigma$—the standard deviation):

$$R_t \sim \mathcal{N}(\mu, \sigma^2)$$

By running thousands of simulations ($N$), we generate a distribution of possible ending wealth outcomes rather than a single misleading average.

### Step 1: Define Your Financial Parameters
Here is where you set the core assumptions for your retirement model. Change these numbers to match your personal scenario!

In [28]:
# Core Simulation Inputs
starting_portfolio = 540000.0  # Initial nest egg ($)
annual_contribution = 25000.0   # Amount added to savings each year ($)
time_horizon_years = 23       # Duration of the projection (Years)
expected_return = 0.075         # Mean annual market return (7.5%)
volatility = 0.15               # Annual market volatility / standard deviation (15%)
num_simulations = 1000          # Number of random future paths to simulate

print(f"Starting Portfolio: ${starting_portfolio:,.2f}")
print(f"Annual Contribution: ${annual_contribution:,.2f} over {time_horizon_years} years.")

Starting Portfolio: $540,000.00
Annual Contribution: $25,000.00 over 23 years.


### Step 2: Generate Random Market Returns
Using our expected return ($\mu$) and volatility ($\sigma$), we draw a grid of random annual returns for every year and every simulation path using a normal distribution.

In [32]:
# Draw random annual returns from a normal distribution
# Shape of matrix: (years, number of simulations)
random_returns = np.random.normal(
    loc=expected_return, 
    scale=volatility, 
    size=(time_horizon_years, num_simulations)
)

print(f"Generated a returns matrix of shape: {random_returns.shape}")
print(f"Sample random returns for Year 1 (first 5 sims): {np.round(random_returns[0, :5], 4)}")

Generated a returns matrix of shape: (23, 1000)
Sample random returns for Year 1 (first 5 sims): [ 0.146  -0.1173  0.4751  0.2761  0.1327]


### Step 3: Run the Step-by-Step Growth Loop
We initialize our portfolio matrix and step through time year-by-year, applying our growth equation: 
$$V_{t+1} = (V_t + C) \times (1 + R_t)$$

In [33]:
# Initialize portfolio matrix with zeros
# Shape: (time_horizon_years + 1, num_simulations) -> +1 includes Year 0 (today)
portfolio_paths = np.zeros((time_horizon_years + 1, num_simulations))

# Set Year 0 starting value across all simulation columns
portfolio_paths[0, :] = starting_portfolio

# Step year-by-year through the timeline
for t in range(time_horizon_years):
    # Apply contributions and market return for year t
    portfolio_paths[t + 1] = (portfolio_paths[t] + annual_contribution) * (1 + random_returns[t])
    
    # Optional safety floor: prevent portfolio from dropping below zero (bankruptcy)
    portfolio_paths[t + 1] = np.maximum(portfolio_paths[t + 1], 0)

print(f"Simulation complete! Final portfolio matrix shape: {portfolio_paths.shape}")

Simulation complete! Final portfolio matrix shape: (24, 1000)


### Step 4: Previewing the Results
Let's check the median ending wealth outcome across all 1,000 simulations to see how the portfolio grew.

In [34]:
# Extract ending portfolio values (the last row of our matrix)
ending_values = portfolio_paths[-1, :]

median_ending_wealth = np.median(ending_values)
p10_ending_wealth = np.percentile(ending_values, 10)
p90_ending_wealth = np.percentile(ending_values, 90)

print(f"--- {time_horizon_years}-Year Projection Summary ---")
print(f"Conservative (10th Percentile): ${p10_ending_wealth:,.2f}")
print(f"Median (50th Percentile):       ${median_ending_wealth:,.2f}")
print(f"Optimistic (90th Percentile):   ${p90_ending_wealth:,.2f}")

--- 23-Year Projection Summary ---
Conservative (10th Percentile): $1,748,481.04
Median (50th Percentile):       $3,709,843.68
Optimistic (90th Percentile):   $8,002,294.27


### Alternative Visualized Results
To rapidly see how different inputs impact the range of outcomes, set the toggles below

In [35]:
# Define interactive widgets
style = {'description_width': '160px'}
layout = {'width': '400px'}

start_val_widget = widgets.FloatText(value=150000, description='Starting Portfolio ($):', style=style, layout=layout)
contribution_widget = widgets.FloatText(value=15000, description='Annual Contribution ($):', style=style, layout=layout)
years_widget = widgets.IntText(value=25, description='Time Horizon (Years):', style=style, layout=layout)
mean_ret_widget = widgets.FloatSlider(value=0.075, min=0.01, max=0.15, step=0.005, description='Expected Return (μ):', continuous_update=False, style=style, layout=layout)
vol_widget = widgets.FloatSlider(value=0.15, min=0.01, max=0.30, step=0.005, description='Volatility / StdDev (σ):', continuous_update=False, style=style, layout=layout)
sims_widget = widgets.IntSlider(value=1000, min=100, max=5000, step=100, description='Number of Sims:', continuous_update=False, style=style, layout=layout)

# Output widget for the plot
out = widgets.Output()

def update_plot(change=None):
    with out:
        clear_output(wait=True)
        
        # Pull values from widgets
        s_val = start_val_widget.value
        contrib = contribution_widget.value
        yrs = years_widget.value
        mu = mean_ret_widget.value
        sigma = vol_widget.value
        nsims = sims_widget.value
        
        # Run simulation using our engine logic
        returns = np.random.normal(loc=mu, scale=sigma, size=(yrs, nsims))
        portfolio = np.zeros((yrs + 1, nsims))
        portfolio[0] = s_val
        
        for t in range(yrs):
            portfolio[t+1] = (portfolio[t] + contrib) * (1 + returns[t])
            portfolio[t+1] = np.maximum(portfolio[t+1], 0)
        
        # Calculate percentiles (10th, 50th/Median, 90th)
        time_axis = np.arange(yrs + 1)
        p10 = np.percentile(portfolio, 10, axis=1)
        p50 = np.percentile(portfolio, 50, axis=1)
        p90 = np.percentile(portfolio, 90, axis=1)
        
        # Plotting
        plt.figure(figsize=(10, 6))
        
        # Plot a subset of individual paths for visual texture
        plt.plot(time_axis, portfolio[:, :50], color='gray', alpha=0.15, linewidth=0.8)
        
        # Plot percentile bands
        plt.plot(time_axis, p50, color='#1f77b4', linewidth=2.5, label='Median Outcome (50th Percentile)')
        plt.plot(time_axis, p90, color='green', linestyle='--', linewidth=1.5, label='Optimistic (90th Percentile)')
        plt.plot(time_axis, p10, color='crimson', linestyle='--', linewidth=1.5, label='Conservative (10th Percentile)')
        
        plt.title('Monte Carlo Retirement Portfolio Projections', fontsize=14, fontweight='bold', pad=15)
        plt.xlabel('Years into Future', fontsize=11)
        plt.ylabel('Portfolio Value ($)', fontsize=11)
        plt.grid(True, linestyle=':', alpha=0.6)
        plt.legend(loc='upper left', frameon=True)
        plt.ticklabel_format(style='plain', axis='y')
        plt.show()

# Bind widgets to update function
for w in [start_val_widget, contribution_widget, years_widget, mean_ret_widget, vol_widget, sims_widget]:
    w.observe(update_plot, names='value')

# Display dashboard layout
dashboard = widgets.VBox([
    widgets.HTML("<h3>Interactive Simulation Controls</h3>"),
    widgets.HBox([widgets.VBox([start_val_widget, contribution_widget, years_widget]),
                  widgets.VBox([mean_ret_widget, vol_widget, sims_widget])]),
    out
])

display(dashboard)

# Initial render call
update_plot()